In [0]:
# 1) Configuración del proyecto
from pyspark.sql import functions as F

CATALOG = "workspace"
SCHEMA = "credito_riesgo_lab"
TABLA_BRONZE = f"{CATALOG}.{SCHEMA}.clientes_bronze"

N_CLIENTES = 10_000   # nº de clientes ficticios
SEED = 42             # semilla: con la misma semilla, los datos son siempre los mismos (reproducible)

# Creamos el esquema del proyecto si aún no existe (es como una "carpeta" de tablas)
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
print(f"Esquema listo: {CATALOG}.{SCHEMA}")

In [0]:
# 2) Generar los clientes ficticios
# spark.range(n) crea un DataFrame con una columna 'id' (0,1,2,...) → la renombramos a cliente_id
clientes = spark.range(1, N_CLIENTES + 1).withColumnRenamed("id", "cliente_id")

# F.rand(semilla) devuelve un número aleatorio entre 0 y 1.
# Usamos una semilla DISTINTA por variable para que no estén correlacionadas.
SECTORES = ["retail", "industria", "servicios", "agro", "construccion", "tecnologia"]

clientes_ficticios = (
    clientes
    .withColumn("r_ing",  F.rand(SEED + 1))
    .withColumn("r_dti",  F.rand(SEED + 2))
    .withColumn("r_ltv",  F.rand(SEED + 3))
    .withColumn("r_ant",  F.rand(SEED + 4))
    .withColumn("r_edad", F.rand(SEED + 5))
    .withColumn("r_pre",  F.rand(SEED + 6))
    .withColumn("r_sec",  F.rand(SEED + 7))
    .withColumn("r_pag",  F.rand(SEED + 8))
    # Variables numéricas: convertimos el aleatorio (0-1) al rango realista de cada variable
    .withColumn("ingresos",        F.round(600 + F.col("r_ing") * 5000, 2))    # € al mes
    .withColumn("endeudamiento",   F.round(0.05 + F.col("r_dti") * 0.75, 3))   # ratio deuda/ingresos (DTI)
    .withColumn("ltv",             F.round(0.10 + F.col("r_ltv") * 1.00, 3))   # loan-to-value del préstamo
    .withColumn("antiguedad_meses", (F.col("r_ant") * 180).cast("int"))        # meses como cliente
    .withColumn("edad",            (18 + F.col("r_edad") * 47).cast("int"))
    .withColumn("num_prestamos",   (1 + F.col("r_pre") * 4).cast("int"))       # 1 a 4 préstamos
    # Sector: elegimos uno al azar de la lista (element_at empieza en 1, por eso +1)
    .withColumn("sector", F.element_at(
        F.array(*[F.lit(s) for s in SECTORES]),
        (F.col("r_sec") * 6 + 1).cast("int"),
    ))
    # Historial de pagos: categorías con pesos realistas (65% puntual, 25% atrasos, 10% moroso)
    .withColumn("historial_pagos",
        F.when(F.col("r_pag") < 0.65, F.lit("puntual"))
         .when(F.col("r_pag") < 0.90, F.lit("atrasos_ocasionales"))
         .otherwise(F.lit("moroso")))
    # Eliminamos las columnas auxiliares de aleatorios (ya no hacen falta)
    .drop("r_ing", "r_dti", "r_ltv", "r_ant", "r_edad", "r_pre", "r_sec", "r_pag")
)

clientes_ficticios.printSchema()   # printSchema muestra las columnas y sus tipos

In [0]:
# 3) Inyectar "suciedad" a propósito y guardar en BRONZE
# - unionByName: pegamos un 2% de filas duplicadas (como si el sistema de origen reenviara registros)
# - withColumn + when: ponemos ~1% de ingresos a NULL (datos que faltan)
df_bronze = (
    clientes_ficticios
    .unionByName(clientes_ficticios.sample(fraction=0.02, seed=SEED + 99))
    .withColumn("ingresos",
        F.when(F.rand(SEED + 10) < 0.01, F.lit(None)).otherwise(F.col("ingresos")))
    .withColumn("_fecha_ingesta", F.current_timestamp())   # metadato típico de la capa bronze
)

# .mode("overwrite") reemplaza la tabla si ya existe (compatible con Spark Connect/Serverless)
(
    df_bronze.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABLA_BRONZE)
)

n = spark.table(TABLA_BRONZE).count()
print(f"Tabla BRONZE creada: {TABLA_BRONZE} → {n} filas")

In [0]:
# 4) Vista rápida de la capa bronze
# Fíjate: hay duplicados (mismo cliente_id) y algunos ingresos a NULL → eso lo arregla la capa silver
display(spark.table(TABLA_BRONZE).limit(5))

# 00 · Datos sintéticos de clientes — Riesgo de crédito

**Objetivo:** generar 10.000 clientes ficticios con las variables típicas de riesgo de crédito (**ingresos, endeudamiento, LTV, antigüedad, historial de pagos, sector**) y guardarlos en la capa **bronze**: `workspace.credito_riesgo_lab.clientes_bronze`.

Truco didáctico: inyectamos a propósito ~2% duplicados y ~1% nulos, para que la capa *silver* (notebook 01) tenga trabajo de limpieza que hacer.